In [ ]:
from transformers import GPT2Tokenizer
import pandas as pd
import re
import numpy as np
import torch
import torch.nn as nn

# dataOld = pd.read_csv("train 2.csv")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# data = dataOld

from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

...

data/train-00003-of-00004-d243063613e5a0(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [ ]:
dataframe = ds['train'].to_pandas()

In [ ]:
# dataframe

In [ ]:
concatString = "\n".join(
    dataframe["text"][:160000].astype(str)
)

In [ ]:
len(concatString)

143628167

In [ ]:
import re

def clean_wikitext(text):
    # WikiText tokenization artifact
    text = text.replace(" @-@ ", "-")

    # Remove unknown-token markers
    text = text.replace("<unk>", "")

    # Convert escaped newlines if literally stored as \n
    text = text.replace("\\n", "\n")

    # Collapse excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Keep paragraph boundaries
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

dataNew = clean_wikitext(concatString)

In [ ]:
len(dataNew)

143626814

In [ ]:
# pattern = re.compile(
#     r"",
#     flags=re.IGNORECASE | re.DOTALL
# )

# examples = []

# for i, d in enumerate(data['output']):
#     d = str(d)

#     matches = pattern.findall(d)

#     for code in matches:
#         examples.append(
#             "### Instruction:\n\n"
#             + str(data['instruction'][i])
#             + "\n### Output:\n\n"
#             + code.strip()
#             + "\n### End\n"
#         )

# dataNew = "\n".join(examples)

In [ ]:
# examples = []

# for i in range(len(data)):
#     instruction = str(data["instruction"][i])
#     output = str(data["output"][i])
#     input_data = str(data["input"][i])

#     # Handle NaN / no input
#     if input_data == "nan" or input_data == "Not applicable":
#         input_data = "No input required"

#     examples.append(
#         "### Instruction:\n\n"
#         + instruction
#         + "\n\n### Input Required:\n\n"
#         + input_data
#         + "\n\n### Output:\n\n"
#         + output
#         + "\n\n### End\n"
#     )

# dataNew = "\n".join(examples)

In [ ]:
tokenizer.model_max_length = int(1e9)

vocab_size = tokenizer.vocab_size

def encode(text):
    return tokenizer.encode(text, add_special_tokens=False)

def decode(tokens):
    return tokenizer.decode(tokens)

with open("tokens.bin", "wb") as f:

    chunk_size = len(dataNew) // 4

    total_tokens = 0

    for i in range(4):
        start = i * chunk_size

        if i == 3:
            chunk = dataNew[start:]
        else:
            end = (i + 1) * chunk_size
            chunk = dataNew[start:end]

        ids = tokenizer.encode(
            chunk,
            add_special_tokens=False
        )

        ids = np.array(ids, dtype=np.uint16)

        ids.tofile(f)

        total_tokens += len(ids)

        print(
            f"Chunk {i+1}/4 written: "
            f"{len(ids):,} tokens | "
            f"Total: {total_tokens:,}"
        )

print(f"\nDone. Total tokens: {total_tokens:,}")

Chunk 1/4 written: 8,942,228 tokens | Total: 8,942,228
Chunk 2/4 written: 8,896,934 tokens | Total: 17,839,162
Chunk 3/4 written: 8,935,255 tokens | Total: 26,774,417
Chunk 4/4 written: 8,979,222 tokens | Total: 35,753,639

Done. Total tokens: 35,753,639


In [ ]:
# from google.colab import drive
# import torch
# import shutil
# import os

# # Mount Google Drive
# drive.mount('/content/drive')

# data = np.memmap(
#     "/content/drive/MyDrive/tokens.bin",
#     dtype=np.uint16,
#     mode="r"
# )

In [ ]:
data = np.memmap(
    "tokens.bin",
    dtype=np.uint16,
    mode="r"
)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [ ]:
len(data)

35753639

In [ ]:
# instruction_lengths = []
# output_lengths = []
# example_lengths = []

# for i in range(len(data)):
#     instruction = str(dataOld['instruction'].iloc[i])
#     output = str(dataOld['output'].iloc[i])

#     instruction_lengths.append(len(encode(instruction)))
#     output_lengths.append(len(encode(output)))

#     example = (
#         "### Instruction:\n\n"
#         + instruction
#         + "\n### Output:\n\n"
#         + output
#         + "\n### End"
#     )

#     example_lengths.append(len(encode(example)))

# print("Instruction:")
# print("  Average:", sum(instruction_lengths) / len(instruction_lengths))
# print("  Max:", max(instruction_lengths))

# print("\nOutput:")
# print("  Average:", sum(output_lengths) / len(output_lengths))
# print("  Max:", max(output_lengths))

# print("\nComplete example:")
# print("  Average:", sum(example_lengths) / len(example_lengths))
# print("  Max:", max(example_lengths))

In [ ]:
## Context Size and Chunking - Block size is the context length, ie how many past tokens the model sees, and batch_size is how many parallel operations run in the model

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            np.array(data[i:i+block_size], dtype=np.int64)
        )
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(
            np.array(data[i+1:i+block_size+1], dtype=np.int64)
        )
        for i in ix
    ])

    x, y = x.to(device), y.to(device)

    return x, y

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
from torch.nn import functional as F

batch_size = 16
block_size = 512
max_iters = 2000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 144
n_head = 8
n_layer = 12
dropout = 0.0

In [ ]:
class Block(nn.Module):
    """Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embeddings
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Positional embeddings
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head=n_head)
                for _ in range(n_layer)
            ]
        )

        # Final layer norm
        self.ln_f = nn.LayerNorm(n_embd)

        # Language-model head
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # idx and targets are both (B,T) tensors of integers

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)
        # (B,T,C)

        # Position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )
        # (T,C)

        # Add token + positional embeddings
        x = tok_emb + pos_emb
        # (B,T,C)

        # Transformer
        x = self.blocks(x)
        # (B,T,C)

        # Final layer norm
        x = self.ln_f(x)
        # (B,T,C)

        # Convert to vocabulary logits
        logits = self.lm_head(x)
        # (B,T,vocab_size)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, stop_text=None):
      for _ in range(max_new_tokens):

          # Crop context to block size
          idx_cond = idx[:, -block_size:]

          # Get predictions
          logits, loss = self(idx_cond)

          # Focus only on the last time step
          logits = logits[:, -1, :]

          # Convert logits to probabilities
          probs = F.softmax(logits, dim=-1)

          # Sample next token
          idx_next = torch.argmax(logits, dim=-1, keepdim=True)

          # Append token
          idx = torch.cat((idx, idx_next), dim=1)

          # Stop if ### End appears
          if stop_text is not None:
              generated_text = decode(idx[0].tolist())

              if stop_text in generated_text:
                  break

      # MUST be outside the for loop
      return idx

In [ ]:
# old_state = m.state_dict()

# model = GPTLanguageModel()
# m = model.to(device)

# m.load_state_dict(old_state)

In [ ]:
model = GPTLanguageModel()
m = model.to(device)

# checkpoint = torch.load(
#     "gpt_python_checkpoint.pth",
#     map_location=device,
#     weights_only=False
# )

# m.load_state_dict(checkpoint["model_state_dict"])

print(
    sum(p.numel() for p in m.parameters()) / 1e6,
    'M parameters'
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

17.601553 M parameters


In [ ]:
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:

        losses = estimate_loss()

        print(
            f"step {iter}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

step 0: train loss 10.9708, val loss 10.9694
step 100: train loss 4.7063, val loss 4.6927
step 200: train loss 4.1870, val loss 4.1617
step 300: train loss 4.0013, val loss 3.9704
step 400: train loss 3.8125, val loss 3.7856
step 500: train loss 3.6215, val loss 3.5887
step 600: train loss 3.4578, val loss 3.4366
step 700: train loss 3.3371, val loss 3.3116
step 800: train loss 3.2171, val loss 3.1845
step 900: train loss 3.1175, val loss 3.0932
step 1000: train loss 3.0450, val loss 3.0134
step 1100: train loss 2.9742, val loss 2.9453
step 1200: train loss 2.9152, val loss 2.8874
step 1300: train loss 2.8662, val loss 2.8287
step 1400: train loss 2.8239, val loss 2.7957
step 1500: train loss 2.7696, val loss 2.7328
step 1600: train loss 2.7578, val loss 2.7296
step 1700: train loss 2.7016, val loss 2.6659
step 1800: train loss 2.6754, val loss 2.6453
step 1900: train loss 2.6303, val loss 2.6153
step 1999: train loss 2.6068, val loss 2.5751


In [ ]:
# Save final checkpoint
checkpoint_path = "gpt_python_checkpoint.pth"

torch.save({
    "model_state_dict": m.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iter": max_iters,
}, checkpoint_path)

print(f"Saved checkpoint: {checkpoint_path}")

# Automatically download to your computer
from google.colab import files
files.download(checkpoint_path)

Saved checkpoint: gpt_python_checkpoint.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# m.eval()

prompt = "Once upon a time there was a man named "

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

generated = m.generate(
    context,
    max_new_tokens=200,
    stop_text="### End"
)

print(decode(generated[0].tolist()))

Once upon a time there was a man named 

One day, there was a little boy named Timmy. Timmy loved to play with his toys and had a lot of fun. One day, Timmy's mom told him to be careful and not to be mad. 

Timmy was sad and didn't want to play with his toys. He wanted to play with his toys and make his friends. But he didn't want to play with his toys. 

Timmy's mom told him that he was wrong and didn't want to play with him. Timmy didn't want to play with his toys, but he was very sad. He didn't want to play with his toys anymore. 

Timmy's mom told him that he was wrong and he was sad. Timmy was sad and didn't want to play with his toys anymore. He didn't want to play with his toys anymore.

Timmy's mom told him that he was wrong and he was happy. Timmy


In [ ]:
from google.colab import drive
import torch
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Paths
checkpoint_path = "/content/drive/MyDrive/gpt_python_checkpoint.pth"
tokens_src = "tokens.bin"
tokens_dst = "/content/drive/MyDrive/tokens.bin"

# Save model checkpoint
torch.save({
    "model_state_dict": m.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iter": max_iters,
}, checkpoint_path)

print(f"Saved checkpoint: {checkpoint_path}")

# Save tokenized dataset
if os.path.exists(tokens_src):
    shutil.copy(tokens_src, tokens_dst)
    print(f"Saved tokens: {tokens_dst}")
else:
    print("tokens.bin not found")

Mounted at /content/drive
Saved checkpoint: /content/drive/MyDrive/gpt_python_checkpoint.pth
Saved tokens: /content/drive/MyDrive/tokens.bin


In [ ]:
print(decode(encode("Once upon a time there was a boy named Timmy.")))

xb, yb = get_batch("train")
print(decode(xb[0][:200].tolist()))

In [ ]:
print(decode(data[:300].tolist()))